# ⚡ Day 06: Shortest Path — Finding the Cheapest Route

---

## Overview

| Algorithm | Graph Type | Negative Weights? | Time |
|-----------|-----------|-------------------|------|
| **BFS** | Unweighted | N/A | O(V+E) |
| **Dijkstra's** | Weighted (non-negative) | ❌ | O(E log V) |
| **Bellman-Ford** | Weighted (any) | ✅ | O(V·E) |

**Interview reality:** Dijkstra's covers 90% of weighted shortest path problems. BFS handles unweighted. Bellman-Ford for negative weights or K-stop constraints.

### Dijkstra's Intuition

Like a flood filling from the source. We always expand the nearest unvisited node (greedy). Use a **min-heap** to efficiently pick the next closest node.

```
Start at source with distance 0.
Pop closest node from heap.
For each neighbor: if new_dist < known dist → update and push.
Skip already-finalized nodes.
```

---

In [ ]:
from collections import defaultdict
import heapq

## 🎯 Problem 1: Network Delay Time (LC #743)

Given a network of n nodes and weighted directed edges, find the time for a signal from node `k` to reach ALL nodes. Return -1 if impossible.

**This IS Dijkstra's.** Find shortest distance to all nodes, answer is the maximum.

In [ ]:
def networkDelayTime(times, n, k):
    """
    Dijkstra's algorithm. Time: O(E log V), Space: O(V + E)
    """
    graph = defaultdict(list)
    for u, v, w in times:
        graph[u].append((v, w))
    
    # Min-heap: (distance, node)
    heap = [(0, k)]
    dist = {}  # node → shortest distance from k
    
    while heap:
        d, node = heapq.heappop(heap)
        if node in dist:
            continue  # Already finalized this node
        dist[node] = d
        for neighbor, weight in graph[node]:
            if neighbor not in dist:
                heapq.heappush(heap, (d + weight, neighbor))
    
    # Did we reach all n nodes?
    return max(dist.values()) if len(dist) == n else -1

print(networkDelayTime([[2,1,1],[2,3,1],[3,4,1]], 4, 2))  # 2
print(networkDelayTime([[1,2,1]], 2, 2))                    # -1 (can't reach node 1)

**Why `if node in dist: continue`?** A node can be pushed to the heap multiple times with different distances. We only care about the FIRST pop (smallest distance). All subsequent pops are stale.

---

## 🎯 Problem 2: Cheapest Flights Within K Stops (LC #787)

Find the cheapest flight from `src` to `dst` with at most `k` stops.

**Why not pure Dijkstra?** The K-stop constraint means a longer-distance path might be valid while a shorter one isn't (too many stops). 

**Approach: Modified Bellman-Ford.** Run relaxation K+1 times (K stops = K+1 edges max). Each round, we can reach one hop further.

In [ ]:
def findCheapestPrice(n, flights, src, dst, k):
    """
    Bellman-Ford with K+1 rounds.
    Time: O(K * E), Space: O(N)
    """
    # dist[i] = cheapest price to reach node i
    dist = [float('inf')] * n
    dist[src] = 0
    
    # K stops = at most K+1 edges
    for _ in range(k + 1):
        # IMPORTANT: use a copy! Otherwise we use updated values
        # from this round, which could exceed K stops.
        temp = dist[:]
        for u, v, w in flights:
            if dist[u] + w < temp[v]:
                temp[v] = dist[u] + w
        dist = temp
    
    return dist[dst] if dist[dst] != float('inf') else -1

flights = [[0,1,100],[1,2,100],[0,2,500]]
print(findCheapestPrice(3, flights, 0, 2, 1))  # 200 (0→1→2)
print(findCheapestPrice(3, flights, 0, 2, 0))  # 500 (0→2 direct)

**Why copy `dist` each round?** Without the copy, a node updated in this round could be used to update further nodes in the same round — effectively using more hops than allowed. The copy ensures each round only extends paths by exactly one edge.

---

## 🎯 Problem 3: Path with Maximum Probability (LC #1514)

Find the path with highest success probability between two nodes. Edge weights are probabilities (0 to 1), and path probability = product of edge probabilities.

**Trick:** Dijkstra's but with a **max-heap** (negate probabilities). We want the MAXIMUM product path.

In [ ]:
def maxProbability(n, edges, succProb, start, end):
    """
    Modified Dijkstra's for maximum probability path.
    Time: O(E log V), Space: O(V + E)
    """
    graph = defaultdict(list)
    for i, (u, v) in enumerate(edges):
        graph[u].append((v, succProb[i]))
        graph[v].append((u, succProb[i]))
    
    # Max-heap using negative probabilities
    heap = [(-1.0, start)]  # start with probability 1.0
    best = [0.0] * n
    best[start] = 1.0
    
    while heap:
        neg_prob, node = heapq.heappop(heap)
        prob = -neg_prob
        if node == end:
            return prob
        if prob < best[node]:
            continue  # Stale entry
        for neighbor, edge_prob in graph[node]:
            new_prob = prob * edge_prob
            if new_prob > best[neighbor]:
                best[neighbor] = new_prob
                heapq.heappush(heap, (-new_prob, neighbor))
    
    return 0.0  # Can't reach end

print(maxProbability(3, [[0,1],[1,2],[0,2]], [0.5,0.5,0.2], 0, 2))  # 0.25

---

## 🗺️ Shortest Path Pattern Map

```
╔══════════════════════════════════════════════════════════════════════╗
║  SHORTEST PATH DECISION TREE                                        ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  Unweighted graph?  → BFS (level = distance)                       ║
║  Weighted, no negative weights?  → Dijkstra's (min-heap)           ║
║  Negative weights or K-hop limit?  → Bellman-Ford                  ║
║  Want max instead of min?  → Negate weights for max-heap           ║
║  All pairs shortest path?  → Floyd-Warshall O(V³)                  ║
║                                                                      ║
║  DIJKSTRA TEMPLATE:                                                 ║
║    heap = [(0, source)]                                             ║
║    while heap:                                                      ║
║        d, node = heappop(heap)                                      ║
║        if node in visited: continue                                 ║
║        visited[node] = d                                            ║
║        for neighbor, weight in graph[node]:                         ║
║            if neighbor not in visited:                              ║
║                heappush(heap, (d + weight, neighbor))               ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Key Idea |
|---|---------|------------|----------|
| 743 | Network Delay Time | Medium | Classic Dijkstra's |
| 787 | Cheapest Flights K Stops | Medium | Bellman-Ford with K rounds |
| 1514 | Path with Maximum Probability | Medium | Max-heap Dijkstra |
| 1631 | Path with Minimum Effort | Medium | Dijkstra on grid |
| 778 | Swim in Rising Water | Hard | Binary search + BFS or Dijkstra |
| 1091 | Shortest Path in Binary Matrix | Medium | BFS (unweighted grid) |

---

## ✅ Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 Weighted shortest path → Dijkstra's (min-heap)                │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Skip already-visited nodes in Dijkstra (lazy deletion)         │
├──────────────────────────────────────────────────────────────────────┤
│  📌 K-stop constraint → Bellman-Ford with K+1 rounds + copy[]     │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Max path → negate for max-heap in Dijkstra                    │
└──────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Advanced Graph Patterns

MST, bipartite checking, and other less-common but interview-worthy patterns.